In [21]:
!pip install numpy pandas scikit-learn matplotlib seaborn
!pip install torch transformers datasets gradio
!pip install nltk


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [22]:
import pandas as pd
import numpy as np
import re
import pickle
from collections import Counter
from sklearn.model_selection import train_test_split

In [23]:
print("hello")

hello


In [24]:
import re, math, pickle, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

In [25]:

import nltk

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import torch
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /Users/varad/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [26]:
df = pd.read_csv("data/train-balanced-sarcasm.csv")
df = df.rename(columns={
    "comment": "text",
    "label": "label"
})

df = df[["text", "label"]]
df.dropna(inplace=True)

# Keep training interactive; full dataset is very large.
MAX_SAMPLES = 50000
if len(df) > MAX_SAMPLES:
    df = df.sample(n=MAX_SAMPLES, random_state=42).reset_index(drop=True)

print(f"Dataset rows used: {len(df)}")
df.head()

Dataset rows used: 50000


,text,label
0,Yes and the buff it desperately needs,1
1,Hilarious because i visited A2 one weekend las...,0
2,Have you asked them what they want to eat?,0
3,Because that worked out so well with the mater...,1
4,But white people can't be discriminated against!,1


In [27]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)        
    text = re.sub(r'[^a-zA-Z\s]', '', text)    
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['clean_text'] = df['text'].apply(clean_text)
df.head()

,text,label,clean_text
0,Yes and the buff it desperately needs,1,yes buff desperately needs
1,Hilarious because i visited A2 one weekend las...,0,hilarious visited one weekend last fall sat co...
2,Have you asked them what they want to eat?,0,asked want eat
3,Because that worked out so well with the mater...,1,worked well material guidelines google cannot ...
4,But white people can't be discriminated against!,1,white people cant discriminated


In [28]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['clean_text'],
    df['label'],
    test_size=0.1,
    random_state=42
)

In [29]:
from collections import Counter

def tokenize(text):
    return text.split()

all_tokens = []
for text in train_texts:
    all_tokens.extend(tokenize(text))

vocab = Counter(all_tokens)
vocab = {word: i+1 for i, (word, _) in enumerate(vocab.items())}

In [30]:
def encode(text, vocab):
    return [vocab.get(word, 0) for word in text.split()]

train_sequences = [encode(text, vocab) for text in train_texts]
val_sequences = [encode(text, vocab) for text in val_texts]

In [31]:
from torch.nn.utils.rnn import pad_sequence

MAX_LEN = 80  # Truncate long comments to speed up LSTM training.
train_sequences = [torch.tensor(seq[:MAX_LEN], dtype=torch.long) for seq in train_sequences]
val_sequences = [torch.tensor(seq[:MAX_LEN], dtype=torch.long) for seq in val_sequences]

train_padded = pad_sequence(train_sequences, batch_first=True)
val_padded = pad_sequence(val_sequences, batch_first=True)

print(f"train_padded shape: {train_padded.shape}")
print(f"val_padded shape: {val_padded.shape}")

train_padded shape: torch.Size([45000, 80])
val_padded shape: torch.Size([5000, 67])


In [32]:
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = torch.tensor(labels.values)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

In [33]:
train_dataset = TextDataset(train_padded, train_labels)
val_dataset = TextDataset(val_padded, val_labels)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [34]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size+1, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        out = self.fc(hidden[-1])
        return out

In [35]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

model = LSTMModel(len(vocab)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Using device: mps


In [36]:
EPOCHS = 3
LOG_EVERY = 100

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch_idx, (x, y) in enumerate(train_loader, start=1):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % LOG_EVERY == 0:
            avg_loss = total_loss / batch_idx
            print(f"Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx}/{len(train_loader)} | Avg Loss: {avg_loss:.4f}")

    print(f"Epoch {epoch+1}/{EPOCHS} complete | Total Loss: {total_loss:.4f}")

Epoch 1/3 | Batch 100/1407 | Avg Loss: 0.6979
Epoch 1/3 | Batch 200/1407 | Avg Loss: 0.6955
Epoch 1/3 | Batch 300/1407 | Avg Loss: 0.6949
Epoch 1/3 | Batch 400/1407 | Avg Loss: 0.6945
Epoch 1/3 | Batch 500/1407 | Avg Loss: 0.6943
Epoch 1/3 | Batch 600/1407 | Avg Loss: 0.6942
Epoch 1/3 | Batch 700/1407 | Avg Loss: 0.6941
Epoch 1/3 | Batch 800/1407 | Avg Loss: 0.6940
Epoch 1/3 | Batch 900/1407 | Avg Loss: 0.6939
Epoch 1/3 | Batch 1000/1407 | Avg Loss: 0.6938
Epoch 1/3 | Batch 1100/1407 | Avg Loss: 0.6938
Epoch 1/3 | Batch 1200/1407 | Avg Loss: 0.6938
Epoch 1/3 | Batch 1300/1407 | Avg Loss: 0.6937
Epoch 1/3 | Batch 1400/1407 | Avg Loss: 0.6937
Epoch 1/3 complete | Total Loss: 976.0240
Epoch 2/3 | Batch 100/1407 | Avg Loss: 0.6933
Epoch 2/3 | Batch 200/1407 | Avg Loss: 0.6933
Epoch 2/3 | Batch 300/1407 | Avg Loss: 0.6931
Epoch 2/3 | Batch 400/1407 | Avg Loss: 0.6932
Epoch 2/3 | Batch 500/1407 | Avg Loss: 0.6932
Epoch 2/3 | Batch 600/1407 | Avg Loss: 0.6932
Epoch 2/3 | Batch 700/1407 | Avg 